### Import important dependancies 

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
from datetime import datetime
import re

class VogueFashionScraper:
    def __init__(self):
        self.base_url = "https://www.vogue.com"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    def scrape_collection_page(self, url):
        """Scrape the main collection page to get all designer links"""
        response = requests.get(url, headers=self.headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        designers = []
        # Find all designer links - this selector may need adjustment
        designer_links = soup.find_all('a', href=re.compile(r'/fashion-shows/'))
        
        for link in designer_links:
            href = link.get('href')
            if href and '/fashion-shows/' in href:
                full_url = self.base_url + href if not href.startswith('http') else href
                designers.append({
                    'name': link.get_text(strip=True),
                    'url': full_url
                })
        
        return designers
    
    def extract_fashion_keywords(self, text):
        """Extract fashion-related keywords and phrases from text"""
        # Common fashion descriptor patterns
        patterns = [
            r'\b[A-Z][a-z]+(?:\s+[a-z]+){0,2}\s+(?:silk|leather|cotton|wool|denim|lace|velvet)\b',
            r'\b(?:sharply|elegantly|boldly|subtly)\s+(?:tailored|cut|designed|structured)\s+\w+\b',
            r'\b(?:oversized|fitted|loose|tight|flowing)\s+\w+\b',
            r'\b[A-Z][a-z]+(?:\s+[a-z]+)?\s+(?:blazer|coat|dress|skirt|pants|jacket|cape)\b',
            r'\b(?:metallic|sequined|embroidered|printed|striped)\s+\w+\b'
        ]
        
        keywords = set()
        for pattern in patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            keywords.update(matches)
        
        return list(keywords)
    
    def scrape_designer_article(self, url):
        """Scrape individual designer article for fashion keywords"""
        try:
            response = requests.get(url, headers=self.headers)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Extract article content
            article_body = soup.find('article') or soup.find('div', class_=re.compile('article|content'))
            
            if not article_body:
                return None
            
            text = article_body.get_text(separator=' ', strip=True)
            keywords = self.extract_fashion_keywords(text)
            
            return {
                'url': url,
                'text_snippet': text[:500],  # First 500 chars
                'fashion_keywords': keywords,
                'scraped_date': datetime.now().isoformat()
            }
        except Exception as e:
            print(f"Error scraping {url}: {e}")
            return None
    
    def scrape_full_collection(self, collection_url):
        """Scrape entire collection with all designers"""
        print(f"Scraping collection: {collection_url}")
        designers = self.scrape_collection_page(collection_url)
        
        results = []
        for i, designer in enumerate(designers):
            print(f"Scraping {i+1}/{len(designers)}: {designer['name']}")
            article_data = self.scrape_designer_article(designer['url'])
            
            if article_data:
                article_data['designer_name'] = designer['name']
                results.append(article_data)
            
            # Be polite - don't hammer the server
            time.sleep(2)
        
        return results
    
    def save_results(self, results, filename=None):
        """Save results to JSON file"""
        if filename is None:
            filename = f"vogue_scrape_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        
        print(f"Results saved to {filename}")

# Usage example
if __name__ == "__main__":
    scraper = VogueFashionScraper()
    results = scraper.scrape_full_collection(
        "https://www.vogue.com/fashion-shows/spring-2025-ready-to-wear"
    )
    scraper.save_results(results)

Scraping collection: https://www.vogue.com/fashion-shows/spring-2025-ready-to-wear
Scraping 1/421: Image Archive
Scraping 2/421: Latest Shows
Scraping 3/421: Seasons
Scraping 4/421: Designers
Scraping 5/421: Featured
Scraping 6/421: A. Potts
Scraping 7/421: A.L.C.
Scraping 8/421: A.W.A.K.E. Mode
Scraping 9/421: Aaron Esh
Scraping 10/421: Abra
Scraping 11/421: Acne Studios
Scraping 12/421: Adam Lippes
Scraping 13/421: Adeam
Scraping 14/421: AGL
Scraping 15/421: Ahluwalia
Scraping 16/421: Akris
Scraping 17/421: Alaïa
Scraping 18/421: Alainpaul
Scraping 19/421: Alberta Ferretti
Scraping 20/421: Alejandra Alonso Rojas
Scraping 21/421: Alessandra Rich
Scraping 22/421: Alexander Wang
Scraping 23/421: Alexis Mabille
Scraping 24/421: Alice + Olivia
Scraping 25/421: Altuzarra
Scraping 26/421: Ambush
Scraping 27/421: Andreadamo
Scraping 28/421: Andreas Kronthaler for Vivienne Westwood
Scraping 29/421: Ann Demeulemeester
Scraping 30/421: Anna October
Scraping 31/421: Anna Sui
Scraping 32/421: Ann

In [4]:
"""
PINTEREST TRENDS - CORRECT URLS (Updated October 2025)

Pinterest reorganized their trends site. Here are the working URLs:
"""

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import json
from datetime import datetime

# ============================================================================
# CORRECT PINTEREST URLS
# ============================================================================

PINTEREST_URLS = {
    # Main trends tool (requires login for full access)
    'trends_tool': 'https://trends.pinterest.com/',
    
    # Today's top trends (public, no login)
    'today_trends': 'https://www.pinterest.com/today/',
    
    # Annual predictions report (best for trend analysis!)
    'predicts_2025': 'https://business.pinterest.com/pinterest-predicts/',
    'predicts_article': 'https://www.pinterest.com/today/article/pinterest-predicts-2025/123145/',
    
    # Seasonal trend reports (published quarterly)
    'summer_2025': 'https://newsroom.pinterest.com/news/the-2025-pinterest-summer-trend-report/',
    'fall_2025': 'https://newsroom.pinterest.com/news/the-2025-pinterest-fall-trend-report/',
    
    # Business blog with trend insights
    'business_blog': 'https://business.pinterest.com/blog/',
}

class PinterestTrendsScraperFixed:
    """
    Updated scraper for Pinterest's actual working URLs
    """
    
    def __init__(self, headless=True):
        options = Options()
        if headless:
            options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
        
        self.driver = webdriver.Chrome(options=options)
    
    def scrape_today_trends(self):
        """
        Scrape https://www.pinterest.com/today/
        This page shows current trending topics (no login required!)
        """
        url = PINTEREST_URLS['today_trends']
        print(f"Scraping Today's Trends: {url}")
        
        self.driver.get(url)
        time.sleep(5)
        
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        
        trends = []
        
        # Pinterest uses different selectors - these may need adjustment
        # Look for article cards, trend cards, or similar containers
        
        # Try multiple possible selectors
        trend_containers = (
            soup.find_all('div', {'data-test-id': 'pinWrapper'}) or
            soup.find_all('article') or
            soup.find_all('div', class_=lambda x: x and 'Card' in str(x))
        )
        
        for container in trend_containers[:20]:  # Get top 20
            # Extract title
            title_elem = (
                container.find('h2') or 
                container.find('h3') or
                container.find('div', class_=lambda x: x and 'title' in str(x).lower())
            )
            
            if title_elem:
                trend = {
                    'title': title_elem.get_text(strip=True),
                    'type': 'today_trending',
                    'scraped_date': datetime.now().isoformat()
                }
                
                # Try to find description
                desc = container.find('p')
                if desc:
                    trend['description'] = desc.get_text(strip=True)
                
                # Try to find link
                link = container.find('a')
                if link and link.get('href'):
                    trend['link'] = link['href']
                
                trends.append(trend)
        
        return trends
    
    def scrape_predicts_report(self):
        """
        Scrape the annual Pinterest Predicts report
        This is their main trend forecast - BEST source for trend analysis!
        """
        url = PINTEREST_URLS['predicts_article']
        print(f"Scraping Pinterest Predicts 2025: {url}")
        
        self.driver.get(url)
        time.sleep(8)  # Longer wait for heavy page
        
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        
        report_data = {
            'year': 2025,
            'report_type': 'Pinterest Predicts',
            'url': url,
            'trends': [],
            'scraped_date': datetime.now().isoformat()
        }
        
        # Find all headings (trends are usually in H2 or H3)
        headings = soup.find_all(['h2', 'h3'])
        
        for heading in headings:
            trend_text = heading.get_text(strip=True)
            
            # Filter out navigation/metadata headings
            if len(trend_text) > 3 and not any(skip in trend_text.lower() for skip in ['sign in', 'menu', 'search', 'save']):
                # Get description from following paragraph
                next_p = heading.find_next('p')
                description = next_p.get_text(strip=True) if next_p else ""
                
                report_data['trends'].append({
                    'trend_name': trend_text,
                    'description': description[:500]  # Limit length
                })
        
        return report_data
    
    def scrape_seasonal_report(self, season='fall', year=2025):
        """
        Scrape seasonal trend reports
        Options: 'summer', 'fall', 'winter', 'spring'
        """
        url_key = f'{season}_{year}'
        
        if url_key not in PINTEREST_URLS:
            print(f"No URL found for {season} {year}")
            return None
        
        url = PINTEREST_URLS[url_key]
        print(f"Scraping {season.title()} {year} Report: {url}")
        
        self.driver.get(url)
        time.sleep(5)
        
        soup = BeautifulSoup(self.driver.page_source, 'html.parser')
        
        # Pinterest newsroom uses article format
        article = soup.find('article') or soup.find('div', class_='article-content')
        
        if not article:
            # Try to get all text as fallback
            article = soup.find('main') or soup
        
        trends = []
        
        # Find all headings and content
        for heading in article.find_all(['h2', 'h3']):
            trend_name = heading.get_text(strip=True)
            
            # Get associated content
            content = []
            next_elem = heading.find_next_sibling()
            while next_elem and next_elem.name not in ['h2', 'h3']:
                if next_elem.name == 'p':
                    content.append(next_elem.get_text(strip=True))
                next_elem = next_elem.find_next_sibling()
            
            if trend_name:
                trends.append({
                    'trend_name': trend_name,
                    'content': ' '.join(content),
                    'season': season,
                    'year': year
                })
        
        return {
            'season': season,
            'year': year,
            'url': url,
            'trends': trends,
            'scraped_date': datetime.now().isoformat()
        }
    
    def scrape_all_fashion_data(self):
        """
        Comprehensive scrape of all available Pinterest trend data
        """
        all_data = {
            'scrape_date': datetime.now().isoformat(),
            'source': 'Pinterest Trends',
            'data_sources': {}
        }
        
        print("\n" + "="*60)
        print("SCRAPING ALL PINTEREST TREND DATA")
        print("="*60)
        
        # 1. Today's trends
        print("\n[1/3] Scraping today's trends...")
        try:
            today = self.scrape_today_trends()
            all_data['data_sources']['today_trends'] = today
            print(f"✓ Found {len(today)} trending topics")
        except Exception as e:
            print(f"✗ Error: {e}")
            all_data['data_sources']['today_trends'] = []
        
        time.sleep(3)
        
        # 2. Annual Predicts report
        print("\n[2/3] Scraping Pinterest Predicts 2025...")
        try:
            predicts = self.scrape_predicts_report()
            all_data['data_sources']['predicts_2025'] = predicts
            print(f"✓ Found {len(predicts['trends'])} predicted trends")
        except Exception as e:
            print(f"✗ Error: {e}")
            all_data['data_sources']['predicts_2025'] = {}
        
        time.sleep(3)
        
        # 3. Latest seasonal report
        print("\n[3/3] Scraping Fall 2025 seasonal report...")
        try:
            fall = self.scrape_seasonal_report('fall', 2025)
            all_data['data_sources']['fall_2025'] = fall
            if fall:
                print(f"✓ Found {len(fall['trends'])} fall trends")
        except Exception as e:
            print(f"✗ Error: {e}")
            all_data['data_sources']['fall_2025'] = {}
        
        return all_data
    
    def save_results(self, data, filename=None):
        """Save results to JSON"""
        if filename is None:
            filename = f"pinterest_trends_{datetime.now().strftime('%Y%m%d')}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"\n{'='*60}")
        print(f"✓ Results saved to: {filename}")
        print(f"{'='*60}")
    
    def close(self):
        """Close browser"""
        self.driver.quit()

# USAGE EXAMPLE
if __name__ == "__main__":
    print("Pinterest Trends Scraper (Fixed URLs)")
    print("="*60)
    
    scraper = PinterestTrendsScraperFixed(headless=True)
    
    try:
        # Scrape all available data
        all_data = scraper.scrape_all_fashion_data()
        
        # Save results
        scraper.save_results(all_data)
        
        # Print summary
        print("\n" + "="*60)
        print("SUMMARY")
        print("="*60)
        
        for source, data in all_data['data_sources'].items():
            if isinstance(data, list):
                print(f"{source}: {len(data)} items")
            elif isinstance(data, dict) and 'trends' in data:
                print(f"{source}: {len(data['trends'])} trends")
            else:
                print(f"{source}: Data collected")
        
    finally:
        scraper.close()

# ALTERNATIVE: Simple requests approach for newsroom articles
def scrape_pinterest_newsroom_simple(url):
    """
    For Pinterest Newsroom articles, you can use simple requests
    (no Selenium needed!)
    """
    import requests
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Extract article content
    article = soup.find('article')
    
    if article:
        trends = []
        for heading in article.find_all(['h2', 'h3']):
            trend_name = heading.get_text(strip=True)
            if trend_name:
                trends.append(trend_name)
        
        return trends
    
    return None

# Test the simple approach
print("\n\nTesting simple approach on Fall 2025 report...")
fall_url = PINTEREST_URLS['fall_2025']
simple_trends = scrape_pinterest_newsroom_simple(fall_url)
if simple_trends:
    print(f"✓ Found {len(simple_trends)} trends without Selenium!")

    print("Trends:", simple_trends[:5])

Pinterest Trends Scraper (Fixed URLs)

SCRAPING ALL PINTEREST TREND DATA

[1/3] Scraping today's trends...
Scraping Today's Trends: https://www.pinterest.com/today/
✓ Found 0 trending topics

[2/3] Scraping Pinterest Predicts 2025...
Scraping Pinterest Predicts 2025: https://www.pinterest.com/today/article/pinterest-predicts-2025/123145/
✓ Found 0 predicted trends

[3/3] Scraping Fall 2025 seasonal report...
Scraping Fall 2025 Report: https://newsroom.pinterest.com/news/the-2025-pinterest-fall-trend-report/
✓ Found 5 fall trends

✓ Results saved to: pinterest_trends_20251011.json

SUMMARY
today_trends: 0 items
predicts_2025: 0 trends
fall_2025: 5 trends


Testing simple approach on Fall 2025 report...


In [5]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime
import re

class PinterestFashionKeywordExtractor:
    """
    Extract specific fashion keywords from Pinterest reports
    Similar to the Vogue scraper output format
    """
    
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    def scrape_fall_2025_report(self):
        """
        Scrape Pinterest Fall 2025 report and extract fashion keywords
        """
        url = "https://newsroom.pinterest.com/news/the-2025-pinterest-fall-trend-report/"
        
        print(f"Fetching: {url}")
        response = requests.get(url, headers=self.headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get the main article content
        article = soup.find('article') or soup.find('div', class_='post-content')
        
        if not article:
            print("Could not find article content")
            return None
        
        text = article.get_text()
        
        # Extract fashion keywords using the visible search terms
        fashion_data = {
            'report': 'Pinterest Fall 2025 Trends',
            'url': url,
            'scraped_date': datetime.now().isoformat(),
            'categories': []
        }
        
        # Category 1: Preppy Fashion
        preppy_keywords = self.extract_keywords_from_text(text, [
            "women's preppy outfits", "preppy vibes", "new preppy style",
            "male preppy outfits", "navy blue stripes", "2000s preppy aesthetic",
            "classic preppy", "preppy chic outfits", "vintage preppy outfits"
        ])
        
        fashion_data['categories'].append({
            'category': 'Preppy Fashion',
            'keywords': preppy_keywords
        })
        
        # Category 2: Color-Inspired Fashion
        color_keywords = self.extract_keywords_from_text(text, [
            "vanilla latte blonde hair", "coffee brown pants outfit",
            "espresso martini outfit", "matcha outfit", 
            "coffee colour shirt outfit men"
        ])
        
        fashion_data['categories'].append({
            'category': 'Food-Inspired Colors',
            'keywords': color_keywords
        })
        
        # Category 3: Accessories
        accessory_keywords = self.extract_keywords_from_text(text, [
            "analogue watch", "best luxury watches for men",
            "vintage brown watch", "vintage digital watches",
            "vintage luxury watch", "vintage watch for men"
        ])
        
        fashion_data['categories'].append({
            'category': 'Vintage Accessories',
            'keywords': accessory_keywords
        })
        
        # Category 4: 60s Fashion
        sixties_keywords = self.extract_keywords_from_text(text, [
            "60s babydoll", "60s babydoll aesthetic", "60s dolly fashion",
            "60s evening gown", "60s fashion outfits", "60s fashion vintage",
            "60s gowns evening dresses", "1960s evening gown"
        ])
        
        fashion_data['categories'].append({
            'category': '60s Vintage Fashion',
            'keywords': sixties_keywords
        })
        
        # Category 5: Streetwear/Casual
        streetwear_keywords = self.extract_keywords_from_text(text, [
            "patchwork hoodies", "patchwork sweatshirt", "patchwork crewneck",
            "patchwork tee shirt", "patchwork tees"
        ])
        
        fashion_data['categories'].append({
            'category': 'Patchwork Streetwear',
            'keywords': streetwear_keywords
        })
        
        # Category 6: Polka Dots
        polka_keywords = self.extract_keywords_from_text(text, [
            "polka dot outfit", "polka dots aesthetic", "polkadot top",
            "polka dot scarf", "polka dot nails", "polka dot french tip nails"
        ])
        
        fashion_data['categories'].append({
            'category': 'Polka Dot Trend',
            'keywords': polka_keywords
        })
        
        # Category 7: Grunge Beauty
        grunge_keywords = self.extract_keywords_from_text(text, [
            "clean grunge makeup", "natural grunge makeup", "90's grunge makeup",
            "2000s grunge makeup", "soft grunge makeup look", "dark grunge makeup",
            "brown grunge makeup", "messy grunge makeup"
        ])
        
        fashion_data['categories'].append({
            'category': 'Grunge Beauty',
            'keywords': grunge_keywords
        })
        
        # Category 8: Haircuts
        hair_keywords = self.extract_keywords_from_text(text, [
            "asymmetrical pixie bob", "chic pixie", "dark brown pixie haircut",
            "honey blonde pixie cut", "pixie haircut 90s", 
            "short blonde pixie black women"
        ])
        
        fashion_data['categories'].append({
            'category': '90s Pixie Cuts',
            'keywords': hair_keywords
        })
        
        # Category 9: Travel Fashion
        travel_keywords = self.extract_keywords_from_text(text, [
            "autumn europe outfits", "cotswolds outfit", "countryside fashion"
        ])
        
        fashion_data['categories'].append({
            'category': 'Travel & Countryside Fashion',
            'keywords': travel_keywords
        })
        
        return fashion_data
    
    def extract_keywords_from_text(self, text, keyword_list):
        """
        Check which keywords appear in the text and extract them
        """
        found_keywords = []
        text_lower = text.lower()
        
        for keyword in keyword_list:
            # Check if keyword exists in text (case insensitive)
            if keyword.lower() in text_lower:
                found_keywords.append(keyword)
        
        return found_keywords
    
    def scrape_predicts_2025(self):
        """
        Scrape Pinterest Predicts 2025 for fashion keywords
        """
        url = "https://business.pinterest.com/pinterest-predicts/"
        
        print(f"Fetching: {url}")
        response = requests.get(url, headers=self.headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        text = soup.get_text()
        
        # Known Pinterest Predicts 2025 trends from search results
        predicts_data = {
            'report': 'Pinterest Predicts 2025',
            'url': url,
            'scraped_date': datetime.now().isoformat(),
            'trends': []
        }
        
        # Extract trend keywords mentioned in the report
        trend_keywords = [
            "rococo", "rococo frills", "surreal soirees",
            "cherry coded", "dark cherry red", "cherry bedroom",
            "sea witchery", "dark siren makeup", "dark mermaid makeup",
            "wavy wet hair look", "sea inspired nails",
            "aura beauty", "castlecore", "castlecore aesthetic",
            "rebel floats", "cream soda aesthetic", "homemade soda",
            "cottagecore", "y2k fashion", "sustainable fashion"
        ]
        
        found = self.extract_keywords_from_text(text, trend_keywords)
        
        predicts_data['trends'] = found
        
        return predicts_data
    
    def format_like_vogue_output(self, pinterest_data):
        """
        Format Pinterest data to match your Vogue scraper output
        """
        formatted_results = []
        
        if 'categories' in pinterest_data:
            for category in pinterest_data['categories']:
                formatted_results.append({
                    'source': pinterest_data['report'],
                    'url': pinterest_data['url'],
                    'category': category['category'],
                    'fashion_keywords': category['keywords'],
                    'scraped_date': pinterest_data['scraped_date']
                })
        
        return formatted_results
    
    def scrape_all_pinterest_trends(self):
        """
        Scrape all Pinterest trend sources and compile fashion keywords
        """
        all_data = {
            'scrape_date': datetime.now().isoformat(),
            'source': 'Pinterest Fashion Trends',
            'reports': []
        }
        
        print("\n" + "="*60)
        print("SCRAPING PINTEREST FASHION KEYWORDS")
        print("="*60)
        
        # 1. Fall 2025 Report
        print("\n[1/2] Scraping Fall 2025 Report...")
        try:
            fall_data = self.scrape_fall_2025_report()
            if fall_data:
                formatted_fall = self.format_like_vogue_output(fall_data)
                all_data['reports'].extend(formatted_fall)
                
                # Print summary
                total_keywords = sum(len(cat['keywords']) for cat in fall_data['categories'])
                print(f"✓ Extracted {total_keywords} fashion keywords across {len(fall_data['categories'])} categories")
        except Exception as e:
            print(f"✗ Error: {e}")
        
        # 2. Pinterest Predicts 2025
        print("\n[2/2] Scraping Pinterest Predicts 2025...")
        try:
            predicts_data = self.scrape_predicts_2025()
            if predicts_data:
                all_data['reports'].append({
                    'source': predicts_data['report'],
                    'url': predicts_data['url'],
                    'category': 'Annual Predictions',
                    'fashion_keywords': predicts_data['trends'],
                    'scraped_date': predicts_data['scraped_date']
                })
                print(f"✓ Extracted {len(predicts_data['trends'])} trend keywords")
        except Exception as e:
            print(f"✗ Error: {e}")
        
        return all_data
    
    def save_results(self, data, filename=None):
        """Save results in JSON format"""
        if filename is None:
            filename = f"pinterest_fashion_keywords_{datetime.now().strftime('%Y%m%d')}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"\n{'='*60}")
        print(f"✓ Results saved to: {filename}")
        print(f"{'='*60}")
        
        return filename

# USAGE
if __name__ == "__main__":
    scraper = PinterestFashionKeywordExtractor()
    
    # Scrape all Pinterest trends
    results = scraper.scrape_all_pinterest_trends()
    
    # Save results
    filename = scraper.save_results(results)
    
    # Print sample of extracted keywords
    print("\n" + "="*60)
    print("SAMPLE FASHION KEYWORDS EXTRACTED:")
    print("="*60)
    
    for i, report in enumerate(results['reports'][:3]):  # Show first 3
        print(f"\n{i+1}. {report['category']}")
        print(f"   Source: {report['source']}")
        keywords = report['fashion_keywords'][:10]  # Show first 10 keywords
        for kw in keywords:
            print(f"   • {kw}")
        if len(report['fashion_keywords']) > 10:
            print(f"   ... and {len(report['fashion_keywords']) - 10} more")
    
    print(f"\n✓ Total reports: {len(results['reports'])}")
    total_keywords = sum(len(r['fashion_keywords']) for r in results['reports'])
    print(f"✓ Total fashion keywords: {total_keywords}")


SCRAPING PINTEREST FASHION KEYWORDS

[1/2] Scraping Fall 2025 Report...
Fetching: https://newsroom.pinterest.com/news/the-2025-pinterest-fall-trend-report/
Could not find article content

[2/2] Scraping Pinterest Predicts 2025...
Fetching: https://business.pinterest.com/pinterest-predicts/
✓ Extracted 0 trend keywords

✓ Results saved to: pinterest_fashion_keywords_20251011.json

SAMPLE FASHION KEYWORDS EXTRACTED:

1. Annual Predictions
   Source: Pinterest Predicts 2025

✓ Total reports: 1
✓ Total fashion keywords: 0


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
from datetime import datetime
import re

class PinterestAutoKeywordExtractor:
    """
    Automatically extract ALL fashion keywords from Pinterest reports
    without manually specifying them
    """
    
    def __init__(self):
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    def extract_pinterest_search_terms(self, text):
        """
        Pinterest reports list trending searches with growth percentages
        Format: "search term +X%"
        
        This extracts ALL of them automatically!
        """
        # Pattern: text followed by +number%
        # Examples: "preppy vibes +5,597%", "vintage watch +65%"
        pattern = r'([A-Za-z0-9\s\'-]+?)\s*\+[\d,]+%'
        
        matches = re.findall(pattern, text)
        
        # Clean up matches
        keywords = []
        for match in matches:
            cleaned = match.strip()
            # Filter out noise (very short terms, all caps, etc)
            if (len(cleaned) > 3 and 
                not cleaned.isupper() and 
                cleaned[0].isalpha()):
                keywords.append(cleaned)
        
        return keywords
    
    def categorize_keywords(self, keywords):
        """
        Automatically categorize fashion keywords by topic
        """
        categories = {
            'Clothing & Outfits': [],
            'Accessories': [],
            'Hair & Beauty': [],
            'Colors & Materials': [],
            'Vintage & Era-Specific': [],
            'Patterns & Prints': [],
            'Home & Decor': [],
            'Food & Lifestyle': [],
            'Other': []
        }
        
        for keyword in keywords:
            kw_lower = keyword.lower()
            
            # Clothing keywords
            if any(word in kw_lower for word in [
                'outfit', 'dress', 'shirt', 'pants', 'jacket', 'coat',
                'hoodie', 'sweatshirt', 'blazer', 'skirt', 'jeans',
                'sweater', 'cardigan', 'top', 'gown', 'suit', 'fashion'
            ]):
                categories['Clothing & Outfits'].append(keyword)
            
            # Accessories
            elif any(word in kw_lower for word in [
                'watch', 'bag', 'shoe', 'hat', 'scarf', 'belt',
                'jewelry', 'sunglasses', 'necklace', 'bracelet'
            ]):
                categories['Accessories'].append(keyword)
            
            # Hair & Beauty
            elif any(word in kw_lower for word in [
                'hair', 'makeup', 'nail', 'beauty', 'skincare',
                'haircut', 'hairstyle', 'lipstick', 'mascara',
                'pixie', 'bob', 'blonde', 'brunette'
            ]):
                categories['Hair & Beauty'].append(keyword)
            
            # Colors & Materials
            elif any(word in kw_lower for word in [
                'blue', 'red', 'black', 'white', 'green', 'pink',
                'brown', 'grey', 'beige', 'navy', 'color',
                'leather', 'silk', 'cotton', 'wool', 'denim',
                'latte', 'coffee', 'vanilla', 'espresso', 'matcha'
            ]):
                categories['Colors & Materials'].append(keyword)
            
            # Vintage & Era
            elif any(word in kw_lower for word in [
                'vintage', '60s', '70s', '80s', '90s', '2000s',
                'retro', 'classic', 'antique', '1960', '1970',
                'preppy', 'grunge', 'deco'
            ]):
                categories['Vintage & Era-Specific'].append(keyword)
            
            # Patterns
            elif any(word in kw_lower for word in [
                'polka dot', 'stripe', 'floral', 'plaid', 'checkered',
                'patchwork', 'print', 'pattern'
            ]):
                categories['Patterns & Prints'].append(keyword)
            
            # Home & Decor
            elif any(word in kw_lower for word in [
                'decor', 'desk', 'office', 'cubicle', 'tile',
                'interior', 'furniture', 'room', 'kitchen',
                'bathroom', 'aesthetic'
            ]):
                categories['Home & Decor'].append(keyword)
            
            # Food & Lifestyle
            elif any(word in kw_lower for word in [
                'cake', 'cookie', 'bread', 'food', 'recipe',
                'bake', 'dessert', 'travel', 'trip', 'destination'
            ]):
                categories['Food & Lifestyle'].append(keyword)
            
            else:
                categories['Other'].append(keyword)
        
        # Remove empty categories
        return {k: v for k, v in categories.items() if v}
    
    def scrape_pinterest_report(self, url, report_name):
        """
        Scrape any Pinterest report and auto-extract keywords
        """
        print(f"\nScraping: {report_name}")
        print(f"URL: {url}")
        
        response = requests.get(url, headers=self.headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Get main content
        article = soup.find('article') or soup.find('main') or soup
        text = article.get_text()
        
        # Extract all search terms with growth percentages
        all_keywords = self.extract_pinterest_search_terms(text)
        
        print(f"✓ Found {len(all_keywords)} trending search terms")
        
        # Categorize them
        categorized = self.categorize_keywords(all_keywords)
        
        # Build result
        result = {
            'report_name': report_name,
            'url': url,
            'scraped_date': datetime.now().isoformat(),
            'total_keywords': len(all_keywords),
            'all_keywords': all_keywords,
            'categorized_keywords': categorized
        }
        
        return result
    
    def scrape_all_reports(self):
        """
        Scrape all available Pinterest trend reports
        """
        reports_to_scrape = [
            {
                'url': 'https://newsroom.pinterest.com/news/the-2025-pinterest-fall-trend-report/',
                'name': 'Pinterest Fall 2025 Trends'
            },
            {
                'url': 'https://newsroom.pinterest.com/news/the-2025-pinterest-summer-trend-report/',
                'name': 'Pinterest Summer 2025 Trends'
            },
        ]
        
        all_results = {
            'scrape_date': datetime.now().isoformat(),
            'source': 'Pinterest Trend Reports',
            'reports': []
        }
        
        print("="*60)
        print("PINTEREST AUTO KEYWORD EXTRACTOR")
        print("="*60)
        
        for report_info in reports_to_scrape:
            try:
                result = self.scrape_pinterest_report(
                    report_info['url'],
                    report_info['name']
                )
                all_results['reports'].append(result)
            except Exception as e:
                print(f"✗ Error scraping {report_info['name']}: {e}")
        
        return all_results
    
    def format_for_analysis(self, results):
        """
        Format results to match Vogue scraper style for easy analysis
        """
        formatted = []
        
        for report in results['reports']:
            for category, keywords in report['categorized_keywords'].items():
                formatted.append({
                    'source': report['report_name'],
                    'url': report['url'],
                    'category': category,
                    'fashion_keywords': keywords,
                    'scraped_date': report['scraped_date']
                })
        
        return formatted
    
    def save_results(self, data, filename=None):
        """Save results to JSON"""
        if filename is None:
            filename = f"pinterest_auto_keywords_{datetime.now().strftime('%Y%m%d')}.json"
        
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        
        print(f"\n{'='*60}")
        print(f"✓ Saved to: {filename}")
        return filename
    
    def print_summary(self, results):
        """Print a nice summary of extracted keywords"""
        print("\n" + "="*60)
        print("EXTRACTION SUMMARY")
        print("="*60)
        
        total_keywords = 0
        
        for report in results['reports']:
            print(f"\n📊 {report['report_name']}")
            print(f"   Total keywords: {report['total_keywords']}")
            
            if 'categorized_keywords' in report:
                print(f"   Categories: {len(report['categorized_keywords'])}")
                
                for category, keywords in report['categorized_keywords'].items():
                    print(f"\n   {category} ({len(keywords)} items):")
                    # Show first 5 examples
                    for kw in keywords[:5]:
                        print(f"      • {kw}")
                    if len(keywords) > 5:
                        print(f"      ... and {len(keywords) - 5} more")
            
            total_keywords += report['total_keywords']
        
        print(f"\n{'='*60}")
        print(f"🎉 TOTAL FASHION KEYWORDS EXTRACTED: {total_keywords}")
        print(f"{'='*60}")

# USAGE
if __name__ == "__main__":
    scraper = PinterestAutoKeywordExtractor()
    
    # Scrape all reports
    results = scraper.scrape_all_reports()
    
    # Print summary
    scraper.print_summary(results)
    
    # Save raw results
    filename = scraper.save_results(results)
    
    # Also save formatted version (matches Vogue output style)
    formatted = scraper.format_for_analysis(results)
    formatted_filename = filename.replace('.json', '_formatted.json')
    
    with open(formatted_filename, 'w', encoding='utf-8') as f:
        json.dump(formatted, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Also saved formatted version: {formatted_filename}")
    
    print("\n💡 TIP: The formatted version matches your Vogue scraper output!")
    print("   You can combine them for unified analysis.")